# Lab 6 — Excel Group Activity

**Day 01 · Data Science Introduction · Cisco AI/ML Training**

---

## Learning objectives

1. Complete a **group Excel analysis** on `team_sales.csv` (pivot, totals, chart).
2. Reproduce the same results in **pandas** (verification, not a separate "checkpoint file").
3. Compare **total Q2** vs **% growth** rankings by region.
4. Present one slide: *Which region won Q2? Which region improved fastest?*

> **Checkpoints:** **20** teams · **4** regions · total Q2 **3006** · growth teams **15** · top region **North**

<!-- cisco-day01-expanded-2026 -->

**Workflow:** Excel first (group) → this notebook to verify.

## Why this matters

Most stakeholders still live in **Excel**. Data scientists must translate between pivot tables and pandas — and spot when totals and growth rates tell **different stories**.

## Excel workflow (mirror in this notebook)

| Step | Excel | pandas equivalent |
|------|-------|-------------------|
| 1 | Open CSV | `read_csv` |
| 2 | Insert → PivotTable | `pivot_table` / `groupby` |
| 3 | Sum Q1 & Q2 by region | `.sum()` |
| 4 | Count growth teams | `(q2 > q1).sum()` |
| 5 | Chart | `matplotlib` bar chart |

---

## 1. Load team sales

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-01":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "hands-on" / "day-01" / "data" / "team_sales.csv").is_file():
            GH_ROOT = parent
            break

TEAM_SALES_CSV = GH_ROOT / "hands-on" / "day-01" / "data" / "team_sales.csv"
df = pd.read_csv(TEAM_SALES_CSV)

print(f"teams: {df['team'].nunique()}")
print(f"regions: {df['region'].nunique()}")
display(df)


---

## 2. Pivot — Q1 vs Q2 by region (Excel PivotTable)

In [ ]:
pivot = df.pivot_table(
    index="region", values=["q1_sales", "q2_sales"], aggfunc="sum"
).astype(int)
pivot["growth_units"] = pivot["q2_sales"] - pivot["q1_sales"]
pivot["growth_rate"] = (pivot["q2_sales"] - pivot["q1_sales"]) / pivot["q1_sales"]
display(pivot.round(3))

print("Rank by Q2 total:", pivot["q2_sales"].sort_values(ascending=False).index.tolist())
print("Rank by % growth:", pivot["growth_rate"].sort_values(ascending=False).index.tolist())


### 2b. `groupby` equivalent (same numbers)

In [ ]:
regional = df.groupby("region").agg(
    q1_sales=("q1_sales", "sum"),
    q2_sales=("q2_sales", "sum"),
).astype(int)
regional["growth"] = regional["q2_sales"] - regional["q1_sales"]
display(regional)


---

## 3. Growth count and top region

In [ ]:
teams_with_growth = int((df["q2_sales"] > df["q1_sales"]).sum())
total_q2 = int(df["q2_sales"].sum())
top_region = regional["q2_sales"].idxmax()

print(f"Teams with Q2 > Q1: {teams_with_growth}")
print(f"Total Q2 sales:     {total_q2}")
print(f"Top region (Q2 $):  {top_region}")


### 3b. Team-level growth table (for Excel conditional formatting)

In [ ]:
team_view = df.copy()
team_view["grew"] = team_view["q2_sales"] > team_view["q1_sales"]
team_view["delta"] = team_view["q2_sales"] - team_view["q1_sales"]
display(team_view.sort_values("delta", ascending=False))
print(f"Declining teams: {(~team_view['grew']).sum()}")


---

## 4. Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors = ["#d62728" if r == top_region else "#1f77b4" for r in regional.index]
regional["q2_sales"].plot(kind="bar", ax=axes[0], color=colors)
axes[0].set_title("Regional Q2 sales (total)")
axes[0].set_ylabel("Q2 sales")
axes[0].tick_params(axis="x", rotation=0)

pivot[["q1_sales", "q2_sales"]].plot(kind="bar", ax=axes[1], width=0.8)
axes[1].set_title("Q1 vs Q2 by region")
axes[1].set_ylabel("Sales")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


### 4b. Growth rate chart — the "second story"

In [ ]:
pivot["growth_rate"].sort_values().plot(kind="barh", figsize=(6, 3), color="seagreen")
plt.title("% growth Q1→Q2 by region")
plt.xlabel("growth rate")
plt.tight_layout()
plt.show()
print("North wins total Q2 but not highest % growth — discuss in your presentation.")


### 4c. Excel step-by-step (do this in your group first)

1. Open `team_sales.csv` in Excel.
2. **Insert → PivotTable** → rows = `region`, values = Sum of `q1_sales`, Sum of `q2_sales`.
3. Add helper column `Grew` = `IF(q2>q1,1,0)` → `SUMIF` by region.
4. Insert **column chart** of regional Q2 totals.
5. Screenshot for your slide — then verify below.

### 4d. SUMIF-style verification in pandas

In [ ]:
for reg in sorted(df["region"].unique()):
    sub = df.loc[df["region"] == reg]
    print(
        f"{reg:6s}  Q1={sub['q1_sales'].sum():4d}  Q2={sub['q2_sales'].sum():4d}  "
        f"growth_teams={(sub['q2_sales'] > sub['q1_sales']).sum()}/5"
    )


### 4e. North vs East — prepare your talking point

In [ ]:
story = pd.DataFrame({
    "region": ["North", "East"],
    "q2_total": [regional.loc["North", "q2_sales"], regional.loc["East", "q2_sales"]],
    "pct_growth": [pivot.loc["North", "growth_rate"], pivot.loc["East", "growth_rate"]],
}).round(3)
display(story)
print("North: highest Q2 total. East: highest % growth. Both can be true.")


### 4f. Copy table to Excel (manual)

In [ ]:
print("Copy this table into Excel for chart formatting:")
display(pivot.round(3))


---

## 5. Verify your Excel answers

In [ ]:
expected = {
    "East": (688, 757, 69),
    "North": (753, 778, 25),
    "South": (698, 722, 24),
    "West": (686, 749, 63),
}

assert len(df) == 20
assert df["region"].nunique() == 4
assert total_q2 == 3006
assert teams_with_growth == 15
assert top_region == "North"

for region_name, (q1, q2, growth) in expected.items():
    row = regional.loc[region_name]
    assert int(row["q1_sales"]) == q1
    assert int(row["q2_sales"]) == q2
    assert int(row["growth"]) == growth

print("✓ Excel answers match Python verification")


## Group presentation prompt

Prepare **one slide** with:
- Regional Q2 totals (bar chart screenshot from Excel or this notebook)
- One sentence: *North had highest Q2 total, but East had highest % growth*
- Which CRISP-DM phase was this analysis? (**Evaluation** / communication)

## Reflection questions

1. When would leadership care more about **% growth** than **absolute Q2**?
2. What would you do differently with 200 teams instead of 20?
3. How does this activity connect to Day 2 Zomato EDA?